In [1]:
import time
import numpy as np

In [2]:
import dask
import dask.array

In [11]:
import h5py
import os
from datetime import datetime

In [21]:
from scipy import ndimage

In [7]:
sample = '024_large_02_3uls'
scan = '0004'
toppath = '/data/visitor/ihmi1620/id19/20261006'

# input paths
procpath = os.path.join(toppath, 'PROCESSED_DATA')
samplepath = os.path.join(procpath, sample, sample+'_'+scan)
int16_volumepath = os.path.join(samplepath, 'reconstructed_volumes','cast_volume')

In [8]:
def load_im_dask(datapath, name, ts=None):
    splitname = name.split('_')
    ts = int(splitname[-3][:-3])
    entry = '/entry'+f'{ts:04d}'+'/reconstruction/results/data'
    tsentry =  '/entry'+f'{ts:04d}'+'/reconstruction/configuration/acquisition/end_time'
    # print(name, entry, tsentry)
    path = os.path.join(datapath, name)
    file = h5py.File(path)
    da = dask.array.from_array(file[entry])
    byte_data = file[tsentry][()]
    timestamp = datetime.fromisoformat(byte_data.decode()).timestamp() # ChatGPT
    return da, timestamp

In [9]:
def stack_4D(datapath):
    names = os.listdir(datapath)
    names.sort()
    data = []
    timestamps = []
    for name in names:
        da, timestamp = load_im_dask(datapath, name)
        data.append(da)
        timestamps.append(timestamp)
    timestamps = np.array(timestamps)
    Stack = dask.array.stack(data, axis=3)
    return Stack, names, timestamps

In [12]:
Stack, names, timestamps = stack_4D(int16_volumepath)

In [14]:
im = Stack[:,:,:,30]

In [16]:
im = im.compute()

In [17]:
im.shape

(3008, 1024, 1024)

In [18]:
subim = im[1000:1600,200:600,200:600]

In [25]:
t0 = time.time()
grad = ndimage.gaussian_filter(subim, 1)
grad = np.gradient(grad, axis = 2)
print(time.time()-t0)

1.2445666790008545


In [24]:
t0 = time.time()
grad = ndimage.gaussian_filter(subim, 1, order=(0, 0, 1), mode="nearest")
print(time.time()-t0)

1.0748894214630127
